In [2]:
import json
from pathlib import Path

with open(Path("../data/site_removed_complexes.json"), "r") as f:
    ligand_interactions = json.load(f)



In [3]:
from Bio.PDB import PDBParser, NeighborSearch
import numpy as np

def calculate_center_of_mass(atoms):
    coords = np.array([atom.coord for atom in atoms])
    center_of_mass = np.mean(coords, axis=0)
    return center_of_mass

def find_pockets_and_centers(structure, radius=6.0):
    atoms = list(structure.get_atoms())
    ns = NeighborSearch(atoms)
    
    pockets = []
    
    for atom in atoms:
        neighbors = ns.search(atom.coord, radius, level='A')
        pocket_atoms = [n for n in neighbors if n.element != 'H']
        
        if len(pocket_atoms) > 0:
            center_of_mass = calculate_center_of_mass(pocket_atoms)
            chain_ids = set([atom.get_parent().get_parent().id for atom in pocket_atoms])
            pockets.append((center_of_mass, chain_ids))
    
    return pockets

def analyze_ligand_protein_interactions(file_path, radius=6.0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('pdb_structure', file_path)
    
    pockets = find_pockets_and_centers(structure, radius=radius)
    
    single_chain_interacting_ligands = []
    interface_interacting_ligands = []
    
    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.id[0] != ' ':
                    ligand_center = calculate_center_of_mass(list(residue.get_atoms()))
                    for pocket_center, chain_ids in pockets:
                        distance = np.linalg.norm(ligand_center - pocket_center)
                        if distance <= radius:
                            if len(chain_ids) == 1:
                                single_chain_interacting_ligands.append((residue.get_resname(), chain.id, distance))
                            else:
                                interface_interacting_ligands.append((residue.get_resname(), chain.id, distance, chain_ids))
                            break
    
    return single_chain_interacting_ligands, interface_interacting_ligands

file_path = '/mnt/ligandpro/db/PDB/pdb2/processed/pdb8p3w.pdb'
single_chain_ligands, interface_ligands = analyze_ligand_protein_interactions(file_path)

# Вывод результатов
print("Summary of ligand interactions:")
print(f"Total ligands interacting with single chain pockets: {len(single_chain_ligands)}")
print(f"Total ligands interacting with interface pockets (between chains): {len(interface_ligands)}\n")

print("Ligands interacting with single chain pockets:")
for ligand, chain_id, distance in single_chain_ligands:
    print(f"Ligand {ligand} in chain {chain_id} interacts with a single chain pocket within {distance:.2f} Å")

print("\nLigands interacting with interface pockets:")
for ligand, chain_id, distance, interacting_chains in interface_ligands:
    chains = ', '.join(interacting_chains)
    print(f"Ligand {ligand} in chain {chain_id} interacts with a pocket between chains {chains} within {distance:.2f} Å")


Summary of ligand interactions:
Total ligands interacting with single chain pockets: 4
Total ligands interacting with interface pockets (between chains): 10

Ligands interacting with single chain pockets:
Ligand PLM in chain A interacts with a single chain pocket within 5.62 Å
Ligand PLM in chain B interacts with a single chain pocket within 5.85 Å
Ligand PLM in chain C interacts with a single chain pocket within 5.69 Å
Ligand PLM in chain D interacts with a single chain pocket within 5.99 Å

Ligands interacting with interface pockets:
Ligand OLC in chain A interacts with a pocket between chains A, E, D within 5.84 Å
Ligand OLC in chain B interacts with a pocket between chains A, B within 5.95 Å
Ligand POV in chain B interacts with a pocket between chains F, B within 4.87 Å
Ligand OLC in chain C interacts with a pocket between chains C, B within 5.39 Å
Ligand OLC in chain D interacts with a pocket between chains D, C within 5.94 Å
Ligand POV in chain D interacts with a pocket between c

In [ ]:
надо разбить на файлы 